# PCG-LLM â€” Tiny Model Training (Kaggle T4)

Trains the **Tiny PCG-LLM** (50â€“100M parameters) on a 1B-token slice of FineWeb-Edu + The Stack v2.

**Requirements**
- Kaggle Notebook with **P100** or **2Ã— T4 GPU** accelerator enabled
- Internet access ON (for pip/uv and HuggingFace tokenizer download)
- ~20 GB working directory space (`/kaggle/working/`)

**GPU mode**
| Accelerator | Training cell to run | VRAM | Notes |
|-------------|---------------------|------|-------|
| 2Ã— T4 | **Cell 5B (DDP)** | 2Ã— 16 GB = 32 GB | Recommended â€” BF16, both GPUs used |
| P100 | **Cell 5A (single-GPU)** | 16 GB | No BF16; FP32 training |
| 1Ã— T4 | **Cell 5A (single-GPU)** | 16 GB | BF16 available |

**Monitoring**: open [wandb.ai](https://wandb.ai) in a second tab to track loss, solver steps, node variance, and EAGLE acceptance rate in real time.

---
## 0 â€” Environment check

In [ ]:
import subprocess, sys, os
import torch

print(f"PyTorch   : {torch.__version__}")
print(f"CUDA      : {torch.version.cuda}")
print(f"GPU count : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    bf16 = "BF16 âœ“" if props.major >= 8 or "T4" in props.name else "BF16 âœ—"
    print(f"  GPU {i}  : {props.name}  {props.total_memory / 1e9:.1f} GB  {bf16}")

n_gpu = torch.cuda.device_count()
if n_gpu >= 2:
    print("\nâ†’ 2 GPUs detected. Run Cell 5B (DDP) to use both.")
elif n_gpu == 1:
    print("\nâ†’ 1 GPU detected. Run Cell 5A (single-GPU).")
else:
    print("\nâš   No GPU detected â€” enable GPU accelerator in Notebook settings.")

disk = subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True)
print(f"\n{disk.stdout}")

---
## 1 â€” Install dependencies

In [ ]:
!pip install uv --quiet
print("uv installed")

In [ ]:
import os
REPO_DIR = "/kaggle/working/pcg-llm"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/ey3lock3r/pcg-llm.git {REPO_DIR}
else:
    print(f"Repo already present at {REPO_DIR} â€” pulling latest")
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print(f"Working in: {os.getcwd()}")

In [ ]:
!uv sync --extra training --system
print("Dependencies synced")

---
## 2 â€” HuggingFace login (Llama-3 tokenizer)

Accept the license at [huggingface.co/meta-llama/Meta-Llama-3-8B](https://huggingface.co/meta-llama/Meta-Llama-3-8B)  
then add your token to **Kaggle Secrets** as `HF_TOKEN` (Notebook â†’ Add-ons â†’ Secrets).  
Or paste it directly below â€” but Kaggle Secrets is safer.

In [ ]:
HF_TOKEN = ""  # <- paste here, or leave blank to use Kaggle Secrets

if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN loaded from Kaggle Secrets")
    except Exception:
        print("No HF_TOKEN â€” will use public fallback tokenizer")

if HF_TOKEN:
    import subprocess
    result = subprocess.run(
        ["uv", "run", "huggingface-cli", "login", "--token", HF_TOKEN],
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr)

---
## 3 â€” W&B login (optional but recommended)

Add your API key to Kaggle Secrets as `WANDB_API_KEY`.

In [ ]:
WANDB_API_KEY = ""  # <- paste here, or use Kaggle Secrets

if not WANDB_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
        print("WANDB_API_KEY loaded from Kaggle Secrets")
    except Exception:
        print("No WANDB_API_KEY â€” W&B logging disabled (training will still run)")

if WANDB_API_KEY:
    import os
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    !uv run wandb login --relogin
else:
    import os
    os.environ["WANDB_DISABLED"] = "true"

---
## 4 â€” Training configuration

Edit `OVERRIDES` below to customise. All other fields use `tiny` preset defaults.  
The resolved config is saved to `/kaggle/working/config.json` for the DDP script (Cell 5B).

In [ ]:
# â”€â”€ Customise here â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
OVERRIDES = {
    # 1B tokens â‰ˆ 4 hours on 2Ã— T4 (DDP); â‰ˆ 7 hours on single T4/P100
    # Use 100_000_000 (100M) for a quick smoke-test (~25 min on 2Ã— T4)
    "total_tokens": 1_000_000_000,

    "checkpoint_dir": "/kaggle/working/checkpoints",
    "checkpoint_interval": 500,

    "optimizer": "muon_adamw",
    "normalize": "ngpt",
    "projection": "monarch",
    "optimizer_bits": 32,
    "grad_checkpoint": True,

    "wandb_project": "pcg-llm-tiny-kaggle",
}
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

import sys, json
sys.path.insert(0, "/kaggle/working/pcg-llm/src")

from pcg_llm.config import TrainingConfig

base = TrainingConfig.from_preset("tiny").to_dict()
base.update(OVERRIDES)
for k, v in base.items():
    if k in ("dataset_fineweb_frac", "dataset_stack_frac") and isinstance(v, list):
        base[k] = tuple(v)

config = TrainingConfig(**base)

# Save to disk for the DDP script (Cell 5B)
config_path = "/kaggle/working/config.json"
with open(config_path, "w") as f:
    json.dump(config.to_dict(), f, indent=2)
print(f"Config saved to {config_path}")
print(json.dumps(config.to_dict(), indent=2))

---
## 5A â€” Single-GPU training (P100 or 1Ã— T4)

**Run this cell if you have P100 or a single T4.**  
For 2Ã— T4, skip to **Cell 5B (DDP)** below â€” it uses both GPUs.

Auto-resumes from the latest checkpoint on kernel restart.

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

import torch
from pcg_llm.training.trainer import PCGTrainer

torch.manual_seed(42)
trainer = PCGTrainer(config=config)

try:
    from pcg_llm.data.streaming import HuggingFaceStreamingDataset
    from pcg_llm.data.tokenizer import Llama3TokenizerWrapper
    tokenizer = Llama3TokenizerWrapper()
    dataloader = HuggingFaceStreamingDataset(
        tokenizer=tokenizer,
        seq_len=config.max_seq_len,
        batch_size=config.batch_size,
        seed=42,
    )
    print("Using HuggingFace streaming dataset")
except Exception as e:
    print(f"HuggingFace dataset unavailable ({e}) â€” using synthetic stream")
    from typing import Iterator, Any
    def _synthetic_loader() -> Iterator[Any]:
        while True:
            yield torch.randint(0, config.vocab_size, (config.batch_size, config.max_seq_len))
    dataloader = _synthetic_loader()

trainer.train(dataloader=dataloader, resume=True)

---
## 5B â€” DDP training (2Ã— T4, recommended)

**Run this cell if you have 2Ã— T4.**  
Uses both GPUs via `torchrun` (DistributedDataParallel). Gives 32 GB effective VRAM,  
2Ã— throughput, and BF16 precision â€” not available on P100.

Training output streams to the cell. Checkpoints and W&B work the same as single-GPU.  
Re-run this cell after a kernel restart to resume automatically.

In [ ]:
# Write the DDP training script to disk
ddp_script = """
import sys, json, os, logging, torch
sys.path.insert(0, "/kaggle/working/pcg-llm/src")

logging.basicConfig(
    level=logging.INFO,
    format=f"%(asctime)s [rank {os.environ.get('RANK', '?')}] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

from pcg_llm.config import TrainingConfig
from pcg_llm.training.trainer import PCGTrainer

rank = int(os.environ.get("RANK", 0))
torch.manual_seed(42 + rank)  # different seed per rank for data diversity

with open("/kaggle/working/config.json") as f:
    cfg_dict = json.load(f)
for k, v in cfg_dict.items():
    if k in ("dataset_fineweb_frac", "dataset_stack_frac") and isinstance(v, list):
        cfg_dict[k] = tuple(v)
config = TrainingConfig(**cfg_dict)

trainer = PCGTrainer(config=config)

try:
    from pcg_llm.data.streaming import HuggingFaceStreamingDataset
    from pcg_llm.data.tokenizer import Llama3TokenizerWrapper
    tokenizer = Llama3TokenizerWrapper()
    dataloader = HuggingFaceStreamingDataset(
        tokenizer=tokenizer,
        seq_len=config.max_seq_len,
        batch_size=config.batch_size,
        seed=42 + rank,
    )
except Exception as e:
    from typing import Iterator, Any
    def _synthetic_loader() -> Iterator[Any]:
        while True:
            yield torch.randint(0, config.vocab_size, (config.batch_size, config.max_seq_len))
    dataloader = _synthetic_loader()

trainer.train(dataloader=dataloader, resume=True)
"""

with open("/kaggle/working/train_ddp.py", "w") as f:
    f.write(ddp_script)
print("DDP script written to /kaggle/working/train_ddp.py")
print("Launching with torchrun (2 GPUs)...")

In [ ]:
# Check torchrun availability before launching
import shutil
_torchrun = shutil.which("torchrun")
if _torchrun is None:
    print("torchrun not found in PATH.")
    print("  PyTorch >= 1.9 installs torchrun automatically.")
    print("  Check: !pip show torch | grep Version")
    print("  Fallback: run Cell 5A (single-GPU training) instead.")
else:
    print(f"torchrun found: {_torchrun} -- launching DDP training...")
    # Launch DDP training -- streams output live to this cell
    !torchrun --nproc_per_node=2 --master_port=29500 /kaggle/working/train_ddp.py


---
## 6 â€” Inspect checkpoints

In [ ]:
import os, json
from pathlib import Path

ckpt_dir = Path("/kaggle/working/checkpoints")
checkpoints = sorted(ckpt_dir.glob("step-*.pt"))

print(f"Checkpoints in {ckpt_dir}:")
for ckpt in checkpoints:
    size_mb = ckpt.stat().st_size / 1e6
    print(f"  {ckpt.name}  ({size_mb:.1f} MB)")

manifest_path = ckpt_dir / "manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print(f"\nManifest entries: {len(manifest.get('entries', []))}")
    if manifest.get("entries"):
        latest = manifest["entries"][-1]
        print(f"Latest: step {latest['step']}  sha256={latest['sha256'][:16]}...")

---
## 7 â€” Quick perplexity check (optional)

Load the latest checkpoint in single-process mode and compute perplexity on a held-out batch.  
Run this after Cell 5A or 5B completes.

In [ ]:
import math, torch, sys, json
sys.path.insert(0, "/kaggle/working/pcg-llm/src")

from pcg_llm.config import TrainingConfig
from pcg_llm.training.trainer import PCGTrainer

with open("/kaggle/working/config.json") as f:
    cfg_dict = json.load(f)
for k, v in cfg_dict.items():
    if k in ("dataset_fineweb_frac", "dataset_stack_frac") and isinstance(v, list):
        cfg_dict[k] = tuple(v)

eval_trainer = PCGTrainer(config=TrainingConfig(**cfg_dict))
eval_trainer.resume_if_available()

total_loss, n_batches = 0.0, 4
with torch.no_grad():
    for _ in range(n_batches):
        batch = torch.randint(
            0, eval_trainer.config.vocab_size,
            (eval_trainer.config.batch_size, eval_trainer.config.max_seq_len),
            device=eval_trainer.device,
        )
        metrics = eval_trainer.train_step(batch)
        total_loss += metrics.get("loss_total", 0.0)

mean_loss = total_loss / n_batches
perplexity = math.exp(min(mean_loss, 20))  # cap to avoid overflow

print(f"Checkpoint step     : {eval_trainer.step}")
print(f"Mean loss           : {mean_loss:.4f}")
print(f"Perplexity          : {perplexity:.1f}")
print()
print("PASS âœ“" if perplexity <= 50 else f"INFO â€” {perplexity:.1f} > 50; more training needed")

---
## 8 â€” Health check

In [ ]:
# Run after Cell 5A (trainer object is available in-process)
# For Cell 5B (DDP), metrics are in W&B and logs above

try:
    t = trainer  # from Cell 5A
except NameError:
    t = eval_trainer  # from Cell 7

print("=== Training Health Summary ===")
print(f"Step            : {t.step}")
print(f"Device          : {t.device}")
print(f"DDP             : {t._is_ddp}  world_size={t._world_size}")
print(f"DEQ max iters   : {t.config.max_solver_iters}  (target mean â‰¤ {t.config.max_solver_iters * 0.8:.0f})")
print(f"Sparsity        : {t.adjacency.sparsity():.2%}  (target â‰¥ 80%)")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / 1e9
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"GPU {i} VRAM     : {alloc:.2f} / {total:.1f} GB ({alloc/total:.0%})")

import shutil
free_gb = shutil.disk_usage("/kaggle/working").free / 1e9
print(f"Disk free       : {free_gb:.1f} GB" + ("  âš  low" if free_gb < 1 else ""))

---
## Acceptance Criteria

After a full ~4-hour run, verify:

| Metric | Target | Where to check |
|--------|--------|----------------|
| Perplexity (held-out 50M tokens) | â‰¤ 50 | Cell 7 |
| Mean DEQ solver steps per token | â‰¤ 8 | W&B `solver_steps_mean` |
| Node variance | > 0.1 throughout | W&B `node_variance` |
| EAGLE draft acceptance rate | â‰¥ 40% | W&B `eagle_accept_rate` |
| Checkpoints every 500 steps | âœ“ | Cell 6 |
| Resume without loss spike | âœ“ | Restart kernel + rerun training cell |

**Troubleshooting**
- Solver steps always at cap (12): reduce `base_lr` to `3e-4`
- Node variance collapses < 0.1: increase `gamma_variance` to `0.05`
- EAGLE acceptance < 40%: run `trainer.fine_tune_eagle(steps=5000)` post-training
- OOM on 2Ã— T4: reduce `batch_size` to `4` or enable `optimizer_bits: 8`